In [18]:
import csv
import time
from pathlib import Path
from urllib.parse import urljoin

import pymysql
import requests
from bs4 import BeautifulSoup

BASE_URL = "http://www.bspider.top/anjuke/"
HEADERS = {
    "User-Agent": ("Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 "
                   "(KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36")
}
FIELDS = ["title", "house", "address", "struct", "area", "floor", "maketime", "price"]

DATA_DIR = Path("data")
DATA_DIR.mkdir(exist_ok=True)

resp = requests.get(BASE_URL, headers=HEADERS, timeout=30)
print("状态码:", resp.status_code)
resp.raise_for_status()                       # 状态码不是 200 时直接抛出异常
resp.encoding = resp.apparent_encoding        # 让中文不乱码
soup = BeautifulSoup(resp.text, "html.parser")
print("页面标题:", soup.title.get_text(strip=True))
print("HTML 长度:", len(resp.text))

状态码: 200
页面标题: 重庆二手房房产网，重庆二手房出售，重庆买房购房交易信息 - 58安居客
HTML 长度: 280427


In [19]:
# 一把抓住整条房源的容器：div.property，一页应有 60 个
rows = soup.select("div.property")
print("div.property 匹配条数:", len(rows))

row = rows[0]
print("父容器 class:", row.get("class"))
print("容器内 h3 数量:", len(row.select("h3")))
print("相对选择器 h3.property-content-title-name 取到:",
      row.select_one("h3.property-content-title-name").get_text(strip=True))

# 容器开头的 HTML，用来确认它刚好包住一条房源
print(str(row)[:260])

div.property 匹配条数: 60
父容器 class: ['property']
容器内 h3 数量: 1
相对选择器 h3.property-content-title-name 取到: 万科物业自营 南坪六公里金域翰林 三号线精装正三房带家具电器
<div class="property" data-v-31e248e6="" data-v-53a259aa="" tongji_tag="fcpc_ersflist_gzcount"><a class="property-ex" data-action="esf_list" data-cp='{"broker_id":"101348367"}' data-ep='{"exposure":{"esf_list":1,"broker_id":"101348367"}}' data-lego='{"entity_i


In [21]:
def parse_house(row):
    """从一条房源容器中解析 8 个字段，按 title→house→address→struct→area→floor→maketime→price 返回。"""
    title_node = row.select_one("h3.property-content-title-name")
    title = title_node.get_text(strip=True) if title_node else ""

    house_node = row.select_one("p.property-content-info-comm-name")
    house = house_node.get_text(strip=True) if house_node else ""

    # 地址被拆成多个 span，整段取文字，不在标签之间自己加空格
    addr_node = row.select_one("p.property-content-info-comm-address")
    address = addr_node.get_text(strip=True) if addr_node else ""

    # 户型由 6 个 span 拼成：3 / 室 / 2 / 厅 / 1 / 卫
    struct_node = row.select_one("p.property-content-info-attribute")
    struct = struct_node.get_text(strip=True) if struct_node else ""

    # 属性组：先排除 property-content-info-attribute 这个 p，剩下的才是面积/朝向/楼层/年代
    info = row.select_one("div.property-content-info")
    texts = []
    if info is not None:
        for p in info.select("p.property-content-info-text"):
            if "property-content-info-attribute" in (p.get("class") or []):
                continue
            texts.append(p.get_text(strip=True))

    area = texts[0] if len(texts) > 0 else ""       # "89.9㎡"
    floor = texts[2] if len(texts) > 2 else ""      # texts[1] 是朝向，不能取成楼层

    # 年代按特征匹配：节点缺失时保持空字符串，顺序变了也不受影响
    maketime = ""
    for text in texts:
        if "年建造" in text:
            maketime = text
            break

    # 单价用 property-price-average；property-price-total 是总价（万元），不要取错
    price_node = row.select_one("p.property-price-average")
    price = price_node.get_text(strip=True) if price_node else ""

    return {"title": title, "house": house, "address": address, "struct": struct,
            "area": area, "floor": floor, "maketime": maketime, "price": price}

In [22]:
first = parse_house(rows[0])
for key in FIELDS:
    print(f"{key:>9}: {first[key]}")

print("\n本页第 2 条:", parse_house(rows[1])["area"], "|", parse_house(rows[1])["price"])
print("本页最后 1 条:", parse_house(rows[-1])["area"], "|", parse_house(rows[-1])["floor"])

# 面积、单价对应的网页原文，用来核对单位
print("\n面积节点:", rows[0].select_one("div.property-content-info").select("p.property-content-info-text")[1])
print("单价节点:", rows[0].select_one("p.property-price-average"))
print("总价节点(不采):", rows[0].select_one("p.property-price-total").get_text(strip=True))

    title: 万科物业自营 南坪六公里金域翰林 三号线精装正三房带家具电器
    house: 万科金域学府翰林菁华里
  address: 南岸学府大道二塘路59号
   struct: 3室2厅1卫
     area: 89.9㎡
    floor: 低层(共27层)
 maketime: 2019年建造
    price: 16465元/㎡

本页第 2 条: 94.5㎡ | 14815元/㎡
本页最后 1 条: 91.3㎡ | 低层(共32层)

面积节点: <p class="property-content-info-text" data-v-53a259aa="">
                            89.9㎡
                        </p>
单价节点: <p class="property-price-average" data-v-53a259aa="">16465元/㎡</p>
总价节点(不采): 148万


In [23]:
def fetch_soup(url):
    """请求一个页面并返回 BeautifulSoup 对象。"""
    r = requests.get(url, headers=HEADERS, timeout=30)
    r.raise_for_status()
    r.encoding = r.apparent_encoding
    return BeautifulSoup(r.text, "html.parser")


trial = []
current = BASE_URL
for i in range(1, 3):
    page_soup = fetch_soup(current)
    page_rows = page_soup.select("div.property")
    trial.extend(parse_house(x) for x in page_rows)
    next_a = page_soup.select_one("a.next")
    href = next_a.get("href")                       # "?page=2"，只有查询参数
    next_url = urljoin(current, href)               # 补成完整地址
    print(f"第{i}页 {current} 本页{len(page_rows)}条 累计{len(trial)}条")
    print(f"    下一页 href={href} → urljoin 后={next_url}")
    time.sleep(1.2)
    current = next_url

p1 = {tuple(x[f] for f in FIELDS) for x in trial[:60]}
p2 = {tuple(x[f] for f in FIELDS) for x in trial[60:]}
print("\n第1页与第2页完全相同的记录数:", len(p1 & p2))

第1页 http://www.bspider.top/anjuke/ 本页60条 累计60条
    下一页 href=?page=2 → urljoin 后=http://www.bspider.top/anjuke/?page=2
第2页 http://www.bspider.top/anjuke/?page=2 本页60条 累计120条
    下一页 href=?page=3 → urljoin 后=http://www.bspider.top/anjuke/?page=3

第1页与第2页完全相同的记录数: 0


In [24]:
records = []
visited = set()          # 已访问地址，防止打转
stop_reason = ""
page_no = 0
current = BASE_URL
MAX_PAGES = 30           # 安全上限，防止无限循环

start = time.time()
while True:
    if current in visited:
        stop_reason = f"地址重复（{current}），中止"
        break
    visited.add(current)
    page_no += 1

    page_soup = fetch_soup(current)
    page_rows = page_soup.select("div.property")
    if not page_rows:
        stop_reason = f"第{page_no}页解析不到房源，按结构异常中止"
        break

    records.extend(parse_house(x) for x in page_rows)
    print(f"第{page_no}页 {current} 本页{len(page_rows)}条 累计{len(records)}条")

    next_a = page_soup.select_one("a.next")
    if next_a is None:
        stop_reason = "分页区里找不到下一页按钮，按结构异常中止"
        break
    classes = next_a.get("class") or []
    href = next_a.get("href") or ""
    # 末页：按钮失去 next-active，href 退化成 "#"，这两条都要判
    if "next-active" not in classes or href in ("", "#"):
        stop_reason = f"下一页按钮已禁用（class={' '.join(classes)}, href={href}）"
        break

    next_url = urljoin(current, href)
    if not next_url.startswith(BASE_URL):
        stop_reason = f"下一页越出教学站范围（{next_url}），中止"
        break
    if page_no >= MAX_PAGES:
        stop_reason = f"达到安全页数上限 {MAX_PAGES} 页，中止"
        break

    time.sleep(1.2)
    current = next_url

print(f"\n采集结束：{page_no} 页，{len(records)} 条，用时 {time.time()-start:.1f} 秒")
print("停止原因:", stop_reason)

第1页 http://www.bspider.top/anjuke/ 本页60条 累计60条
第2页 http://www.bspider.top/anjuke/?page=2 本页60条 累计120条
第3页 http://www.bspider.top/anjuke/?page=3 本页60条 累计180条
第4页 http://www.bspider.top/anjuke/?page=4 本页60条 累计240条
第5页 http://www.bspider.top/anjuke/?page=5 本页60条 累计300条
第6页 http://www.bspider.top/anjuke/?page=6 本页60条 累计360条
第7页 http://www.bspider.top/anjuke/?page=7 本页60条 累计420条
第8页 http://www.bspider.top/anjuke/?page=8 本页60条 累计480条
第9页 http://www.bspider.top/anjuke/?page=9 本页60条 累计540条
第10页 http://www.bspider.top/anjuke/?page=10 本页60条 累计600条

采集结束：10 页，600 条，用时 31.9 秒
停止原因: 下一页按钮已禁用（class=next, href=#）


In [12]:
# 8 个字段完全相同才算重复
seen = set()
unique_records = []
for rec in records:
    key = tuple(rec[f] for f in FIELDS)
    if key in seen:
        continue
    seen.add(key)
    unique_records.append(rec)

print("去重前:", len(records), "条")
print("去重后:", len(unique_records), "条")
print("完全相同被去掉:", len(records) - len(unique_records), "条")

csv_path = DATA_DIR / "houseinfo.csv"
with open(csv_path, "w", encoding="utf-8-sig", newline="") as f:
    writer = csv.writer(f)
    writer.writerow(FIELDS)                     # 表头与 FIELDS 顺序一致
    for rec in unique_records:
        writer.writerow([rec[k] for k in FIELDS])

print("已写入:", csv_path.resolve(), csv_path.stat().st_size, "字节")

去重前: 600 条
去重后: 498 条
完全相同被去掉: 102 条
已写入: /home/pwuser/workspace/lab2/data/houseinfo.csv 94453 字节


In [13]:
with open(csv_path, encoding="utf-8-sig", newline="") as f:
    reader = csv.DictReader(f)
    header = reader.fieldnames
    back = list(reader)

print("回读表头:", header)
print("回读数据行数(不含表头):", len(back))
print("\n第 1 条:", back[0])
print("\n跨页第 61 条:", back[60])
print("\n最后 1 条:", back[-1])

回读表头: ['title', 'house', 'address', 'struct', 'area', 'floor', 'maketime', 'price']
回读数据行数(不含表头): 498

第 1 条: {'title': '万科物业自营 南坪六公里金域翰林 三号线精装正三房带家具电器', 'house': '万科金域学府翰林菁华里', 'address': '南岸学府大道二塘路59号', 'struct': '3室2厅1卫', 'area': '89.9㎡', 'floor': '低层(共27层)', 'maketime': '2019年建造', 'price': '16465元/㎡'}

跨页第 61 条: {'title': '万科物业自营 朝中庭 业主诚心出售', 'house': '万科金域学府翰林菁华里', 'address': '南岸学府大道二塘路59号', 'struct': '3室2厅1卫', 'area': '85.7㎡', 'floor': '低层(共27层)', 'maketime': '2018年建造', 'price': '16914元/㎡'}

最后 1 条: {'title': '恒大照母山 精装3房 内圈位置 户型方正 拎包即住 诚意出售', 'house': '恒大照母山山水城', 'address': '渝北照母山恒康路9号', 'struct': '3室2厅1卫', 'area': '90.2㎡', 'floor': '低层(共32层)', 'maketime': '2015年建造', 'price': '15530元/㎡'}


In [14]:
with open(csv_path, encoding="utf-8-sig", newline="") as f:
    reader = csv.DictReader(f)
    assert reader.fieldnames == FIELDS, header
    rows_from_csv = [tuple((r[k] or "").strip() for k in FIELDS) for r in reader]
print("从 CSV 读到:", len(rows_from_csv), "条，每条 8 个字段")


def save_to_mysql(rows_data):
    """把 8 字段元组写入 houseinfo，返回(待插入数, 实际写入数)。"""
    conn = None
    cur = None
    try:
        conn = pymysql.connect(host="mysql", port=3306, user="student", password="123456",
                               database="smartpark_lab", charset="utf8mb4", autocommit=False)
        cur = conn.cursor()
        cur.execute("SELECT VERSION()")
        print("已连接 MySQL:", cur.fetchone()[0])

        # 库里已有的 8 个字段，用来避免重复插入
        cur.execute("SELECT title, house, address, struct, area, floor, maketime, price FROM houseinfo")
        existing = set(cur.fetchall())
        to_insert = [r for r in rows_data if r not in existing]
        print("库中已有:", len(existing), "条；本次待插入:", len(to_insert), "条")

        if to_insert:
            sql = ("INSERT INTO houseinfo "
                   "(title, house, address, struct, area, floor, maketime, price) "
                   "VALUES (%s, %s, %s, %s, %s, %s, %s, %s)")
            affected = cur.executemany(sql, to_insert)   # 8 个 %s 对应 8 个字段
            conn.commit()                                # 提交后终端里才查得到
            print("executemany 影响行数:", affected, "，已提交")
        else:
            print("没有新记录，无需插入")
        return len(to_insert), len(to_insert)
    except Exception as e:
        print("写入失败:", repr(e))
        if conn is not None:
            conn.rollback()
            print("已回滚，本次事务未生效")
        return 0, 0
    finally:
        if cur is not None:
            cur.close()
        if conn is not None:
            conn.close()
        print("游标、连接已关闭")


inserted, _ = save_to_mysql(rows_from_csv)

从 CSV 读到: 498 条，每条 8 个字段
已连接 MySQL: 8.4.11
库中已有: 498 条；本次待插入: 0 条
没有新记录，无需插入
游标、连接已关闭


In [15]:
# 用新连接、同一批 CSV 再跑一次，模拟关掉程序后重跑
save_to_mysql(rows_from_csv)

已连接 MySQL: 8.4.11
库中已有: 498 条；本次待插入: 0 条
没有新记录，无需插入
游标、连接已关闭


(0, 0)

In [16]:
conn = pymysql.connect(host="mysql", port=3306, user="student", password="123456",
                       database="smartpark_lab", charset="utf8mb4", autocommit=False)
cur = conn.cursor()

cur.execute("SELECT COUNT(*) FROM houseinfo")
print("数据库总条数:", cur.fetchone()[0])

cur.execute("SELECT id, title, area, floor, maketime, price FROM houseinfo ORDER BY id LIMIT 5")
print("\n前 5 条:")
for r in cur.fetchall():
    print(r)

# 取第一条、跨页一条、最后一条逐字段核对
for rid in (1, 61, 498):
    cur.execute("SELECT title, house, address, struct, area, floor, maketime, price "
                "FROM houseinfo WHERE id=%s", (rid,))
    print(f"\nid={rid}:", cur.fetchone())

cur.close()
conn.close()
print("\n核对完成，连接已关闭")

数据库总条数: 498

前 5 条:
(1, '万科物业自营 南坪六公里金域翰林 三号线精装正三房带家具电器', '89.9㎡', '低层(共27层)', '2019年建造', '16465元/㎡')
(2, '黄泥磅 两代一家 正规三房双卫 业主自住房 拎包入住', '94.5㎡', '低层(共28层)', '2008年建造', '14815元/㎡')
(3, '鸳鸯16年次新小区 大雅云居山刚需经典3房户型 随时看房诚售', '88㎡', '低层(共24层)', '2016年建造', '16487元/㎡')
(4, '中交中央公园 横厅精装大两房 出门都是公园', '91㎡', '低层(共25层)', '2016年建造', '20660元/㎡')
(5, '富力现代广场 豪装大两房 户型方正 看房方便', '85.5㎡', '低层(共30层)', '2012年建造', '16969元/㎡')

id=1: ('万科物业自营 南坪六公里金域翰林 三号线精装正三房带家具电器', '万科金域学府翰林菁华里', '南岸学府大道二塘路59号', '3室2厅1卫', '89.9㎡', '低层(共27层)', '2019年建造', '16465元/㎡')

id=61: ('万科物业自营 朝中庭 业主诚心出售', '万科金域学府翰林菁华里', '南岸学府大道二塘路59号', '3室2厅1卫', '85.7㎡', '低层(共27层)', '2018年建造', '16914元/㎡')

id=498: ('恒大照母山 精装3房 内圈位置 户型方正 拎包即住 诚意出售', '恒大照母山山水城', '渝北照母山恒康路9号', '3室2厅1卫', '90.2㎡', '低层(共32层)', '2015年建造', '15530元/㎡')

核对完成，连接已关闭


In [17]:
!wc -l data/houseinfo.csv


499 data/houseinfo.csv


In [28]:
import os


def q(sql):
    """执行一条 mysql 客户端命令，表格直接打到单元输出里。

    不能写成 print("!mysql ...")，那样只是打印字符串、并不会执行；
    用 get_ipython().system() 才会真的把命令交给 shell 跑。
    密码从环境变量 MYSQL_PWD 取（与 mysql-console.bat 一致），不打印明文。
    """
    sql = " ".join(sql.split())              # 压成一行，避免 -e 里换行
    pwd = os.environ.get("MYSQL_PWD", "123456")
    cmd = (f'MYSQL_PWD={pwd} mysql -h mysql -ustudent smartpark_lab '
           f'--default-character-set=utf8mb4 -t -e "{sql}"')
    print("$ " + cmd.replace(pwd, "***"))    # 回显命令，密码打码
    return get_ipython().system(cmd)


q("""
    show databases;
    use smartpark_lab;
    show tables;
    describe houseinfo;
    select count(*) as before_count from houseinfo;
""")


$ MYSQL_PWD=*** mysql -h mysql -ustudent smartpark_lab --default-character-set=utf8mb4 -t -e "show databases; use smartpark_lab; show tables; describe houseinfo; select count(*) as before_count from houseinfo;"
+--------------------+
| Database           |
+--------------------+
| information_schema |
| performance_schema |
| smartpark_lab      |
+--------------------+
+-------------------------+
| Tables_in_smartpark_lab |
+-------------------------+
| houseinfo               |
+-------------------------+
+----------+--------------+------+-----+---------+----------------+
| Field    | Type         | Null | Key | Default | Extra          |
+----------+--------------+------+-----+---------+----------------+
| id       | int          | NO   | PRI | NULL    | auto_increment |
| title    | varchar(200) | YES  |     | NULL    |                |
| house    | varchar(50)  | YES  |     | NULL    |                |
| address  | varchar(200) | YES  |     | NULL    |                |
| struct   |

In [29]:
q("""
    use smartpark_lab;
    select count(*) as after_count from houseinfo;
    select id, title, area, floor, maketime, price
      from houseinfo order by id limit 5;
""")


$ MYSQL_PWD=*** mysql -h mysql -ustudent smartpark_lab --default-character-set=utf8mb4 -t -e "use smartpark_lab; select count(*) as after_count from houseinfo; select id, title, area, floor, maketime, price from houseinfo order by id limit 5;"
+-------------+
| after_count |
+-------------+
|         498 |
+-------------+
+----+----------------------------------------------------------------------------------------+---------+------------------+---------------+--------------+
| id | title                                                                                  | area    | floor            | maketime      | price        |
+----+----------------------------------------------------------------------------------------+---------+------------------+---------------+--------------+
|  1 | 万科物业自营 南坪六公里金域翰林 三号线精装正三房带家具电器                             | 89.9㎡  | 低层(共27层)     | 2019年建造    | 16465元/㎡   |
|  2 | 黄泥磅 两代一家 正规三房双卫 业主自住房 拎包入住                                       | 94.5㎡  | 低层(共28层

In [30]:
q("""
    select id, title, area, floor, maketime, price
      from houseinfo where id in (1, 61, 498);
""")


$ MYSQL_PWD=*** mysql -h mysql -ustudent smartpark_lab --default-character-set=utf8mb4 -t -e "select id, title, area, floor, maketime, price from houseinfo where id in (1, 61, 498);"
+-----+----------------------------------------------------------------------------------------+---------+------------------+---------------+--------------+
| id  | title                                                                                  | area    | floor            | maketime      | price        |
+-----+----------------------------------------------------------------------------------------+---------+------------------+---------------+--------------+
|   1 | 万科物业自营 南坪六公里金域翰林 三号线精装正三房带家具电器                             | 89.9㎡  | 低层(共27层)     | 2019年建造    | 16465元/㎡   |
|  61 | 万科物业自营 朝中庭 业主诚心出售                                                       | 85.7㎡  | 低层(共27层)     | 2018年建造    | 16914元/㎡   |
| 498 | 恒大照母山 精装3房 内圈位置 户型方正 拎包即住 诚意出售                                 | 90.2㎡  | 低层(共32层)     